In [1]:
# ============================================================
# 02 - Preprocessing
# ============================================================
# Owner: Benjamin Muniz
# Purpose: Clean all datasets, standardize label columns,
#          drop unusable columns, and save the 70/15/15
#          train/val/test splits to data/processed/
#          The URL dataset (PhiUSIIL) is the primary dataset
#          for model training. Email datasets are cleaned and
#          saved for Theryn's content feature extraction.
# ============================================================

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
# ================================================
# Load raw datasets
# ================================================

df_email1 = pd.read_csv('../data/raw/phishing_email_kaggle1.csv')
df_email2 = pd.read_csv('../data/raw/phishing_email_kaggle2.csv')
df_url    = pd.read_csv('../data/raw/phishing_url.csv')

print(f"  Email1 shape: {df_email1.shape}")
print(f"  Email2 shape: {df_email2.shape}")
print(f"  URL shape:    {df_url.shape}")

  Email1 shape: (82486, 2)
  Email2 shape: (18650, 3)
  URL shape:    (235795, 56)


In [4]:
# ================================================
# Clean Email1 — already mostly clean from EDA
# ================================================

# Drop duplicates
df_email1 = df_email1.drop_duplicates()

# Confirm labels are already 0/1
print("Email1 label values:", df_email1['label'].unique())
print("Email1 shape after dedup:", df_email1.shape)
print("Email1 null values:", df_email1.isnull().sum().sum())

Email1 label values: [0 1]
Email1 shape after dedup: (82078, 2)
Email1 null values: 0


In [5]:
# ================================================
# Clean Email2 — needs more work based on EDA
# ================================================

# Drop the unnamed index column
df_email2 = df_email2.drop(columns=['Unnamed: 0'])

# Drop the 16 rows with null email text
df_email2 = df_email2.dropna(subset=['Email Text'])

# Drop duplicates
df_email2 = df_email2.drop_duplicates()

# Standardize label column from strings to 0/1
df_email2['label'] = df_email2['Email Type'].map({
    'Safe Email': 0,
    'Phishing Email': 1
})

# Drop the original string label column now that we have numeric
df_email2 = df_email2.drop(columns=['Email Type'])

# Rename for consistency
df_email2 = df_email2.rename(columns={'Email Text': 'text'})

# Verify
print("Email2 label values:", df_email2['label'].unique())
print("Email2 columns:", df_email2.columns.tolist())
print("Email2 shape after cleaning:", df_email2.shape)
print("Email2 null values:", df_email2.isnull().sum().sum())

Email2 label values: [0 1]
Email2 columns: ['text', 'label']
Email2 shape after cleaning: (17538, 2)
Email2 null values: 0


In [6]:
# ================================================
# Clean URL dataset — drop non-numeric columns
# identified in EDA: FILENAME, URL, Domain, TLD, Title
# ================================================

cols_to_drop = ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']
df_url = df_url.drop(columns=cols_to_drop)

# Drop duplicates
df_url = df_url.drop_duplicates()

# Confirm shape and no nulls
print("URL shape after cleaning:", df_url.shape)
print("URL null values:", df_url.isnull().sum().sum())
print("URL label values:", df_url['label'].unique())
print("URL columns remaining:", df_url.columns.tolist())

URL shape after cleaning: (234987, 51)
URL null values: 0
URL label values: [1 0]
URL columns remaining: ['URLLength', 'DomainLength', 'IsDomainIP', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']


In [7]:
# ================================================
# Final null check across all three datasets
# ================================================

print("=== Final Null Check ===")
print(f"Email1 nulls: {df_email1.isnull().sum().sum()}")
print(f"Email2 nulls: {df_email2.isnull().sum().sum()}")
print(f"URL nulls:    {df_url.isnull().sum().sum()}")

print("\n=== Final Shapes ===")
print(f"Email1: {df_email1.shape}")
print(f"Email2: {df_email2.shape}")
print(f"URL:    {df_url.shape}")

print("\n=== Label Distributions ===")
print("Email1:\n", df_email1['label'].value_counts())
print("\nEmail2:\n", df_email2['label'].value_counts())
print("\nURL:\n",    df_url['label'].value_counts())

=== Final Null Check ===
Email1 nulls: 0
Email2 nulls: 0
URL nulls:    0

=== Final Shapes ===
Email1: (82078, 2)
Email2: (17538, 2)
URL:    (234987, 51)

=== Label Distributions ===
Email1:
 label
1    42845
0    39233
Name: count, dtype: int64

Email2:
 label
0    10980
1     6558
Name: count, dtype: int64

URL:
 label
1    134850
0    100137
Name: count, dtype: int64


In [8]:
# ================================================
# Save cleaned email datasets for Theryn
# These go to data/processed/ for her to use
# ================================================

df_email1.to_csv('../data/processed/email1_clean.csv', index=False)
df_email2.to_csv('../data/processed/email2_clean.csv', index=False)

In [9]:
# ================================================
# Train/Val/Test Split — URL dataset only
# 70% train / 15% validation / 15% test
# stratify=y preserves class balance in all splits
# ================================================

X = df_url.drop('label', axis=1)
y = df_url['label']

# First split off 15% test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=489,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.1765,
    random_state=489,
    stratify=y_temp
)

print("=== Split Sizes ===")
print(f"Train: {X_train.shape[0]} rows ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Val:   {X_val.shape[0]} rows ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} rows ({X_test.shape[0]/len(X)*100:.1f}%)")

print("\n=== Class Balance Per Split ===")
print("Train:\n", y_train.value_counts())
print("\nVal:\n",   y_val.value_counts())
print("\nTest:\n",  y_test.value_counts())

=== Split Sizes ===
Train: 164484 rows (70.0%)
Val:   35254 rows (15.0%)
Test:  35249 rows (15.0%)

=== Class Balance Per Split ===
Train:
 label
1    94391
0    70093
Name: count, dtype: int64

Val:
 label
1    20231
0    15023
Name: count, dtype: int64

Test:
 label
1    20228
0    15021
Name: count, dtype: int64


In [10]:
# ================================================
# Save train/val/test splits to data/processed/
# These are committed to GitHub unlike raw datasets
# ================================================

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_val.to_csv('../data/processed/X_val.csv',     index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_val.to_csv('../data/processed/y_val.csv',     index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)

In [11]:
# ================================================
# Reload and verify splits saved correctly
# ================================================

X_train_check = pd.read_csv('../data/processed/X_train.csv')
y_train_check = pd.read_csv('../data/processed/y_train.csv')

print(f"  X_train shape: {X_train_check.shape}")
print(f"  y_train shape: {y_train_check.shape}")
print(f"  X_train nulls: {X_train_check.isnull().sum().sum()}")
print(f"  y_train nulls: {y_train_check.isnull().sum().sum()}")

  X_train shape: (164484, 50)
  y_train shape: (164484, 1)
  X_train nulls: 0
  y_train nulls: 0


In [12]:
import sys, os
sys.path.append(os.path.abspath('..'))
from src.features import *

# Test URLs
phishing_url = "http://192.168.1.1/secure/login"
legit_url    = "https://www.google.com/search?q=python"
shortener    = "https://bit.ly/3xYz"
phishing_html = '<a href="http://evil.com">Click here to verify your account</a>'
legit_html    = '<a href="https://google.com">Google</a>'

print("=== has_ip_url ===")
print("phishing:", has_ip_url(phishing_url))   # expect 1
print("legit:   ", has_ip_url(legit_url))       # expect 0

print("\n=== max_dots ===")
print("phishing:", max_dots(phishing_url))      # expect 3
print("legit:   ", max_dots(legit_url))         # expect 2

print("\n=== has_url_shortener ===")
print("shortener:", has_url_shortener(shortener))  # expect 1
print("legit:    ", has_url_shortener(legit_url))  # expect 0

print("\n=== num_links ===")
print("phishing html:", num_links(phishing_html))  # expect 1
print("legit html:   ", num_links(legit_html))     # expect 1

print("\n=== num_domains ===")
print("phishing html:", num_domains(phishing_html))  # expect 1
print("legit html:   ", num_domains(legit_html))     # expect 1

print("\n=== has_nonmatching_url ===")
print("phishing html:", has_nonmatching_url(phishing_html))  # expect 0
print("legit html:   ", has_nonmatching_url(legit_html))     # expect 0

=== has_ip_url ===
phishing: 1
legit:    0

=== max_dots ===
phishing: 3
legit:    2

=== has_url_shortener ===
shortener: 1
legit:     0

=== num_links ===
phishing html: 1
legit html:    1

=== num_domains ===
phishing html: 1
legit html:    1

=== has_nonmatching_url ===
phishing html: 0
legit html:    0
